# Continuous Bag-of-Words (CBOW) in PyTorch

This notebook provides a clean, cohesive, and beginner-friendly implementation of the **Continuous Bag-of-Words (CBOW)** word embedding algorithm using PyTorch.
Word embeddings are foundational in Natural Language Processing (NLP) as they map words into a continuous vector space where semantically similar words are close to each other.


## What is CBOW?

The CBOW model was introduced by Mikolov et al. in 2013 as part of the Word2Vec framework. The primary goal of CBOW is to **predict a target word based on its surrounding context words**.

For example, in the sentence:
> *"The cat sat on the mat"*

If the target word is **"sat"** and the context window size is $C = 2$:
- **Context words (Input)**: `["the", "cat", "on", "the"]`
- **Target word (Label)**: `"sat"`

The model works by:
1. Retrieving dense embedding vectors for each of the context words.
2. Averaging (mean pooling) these context embeddings into a single vector.
3. Projecting the averaged vector back to the vocabulary space to produce logits.
4. Comparing the logits to the target word label using Cross-Entropy loss.

Because it averages the context embeddings, it treats the context as a "bag of words"—the order of the context words does not affect the prediction, hence the name Continuous Bag-of-Words.


## Import Libraries

First, we import the necessary libraries. We will use `re` for text cleaning, standard python containers for vocabulary handling, and `torch` alongside its neural network modules (`nn`, `optim`, `DataLoader`) for building and training our CBOW model.


In [1]:
import re
import os
import json
from typing import List, Tuple, Dict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


## Load Dataset

Next, we load the raw text corpus from the local file `sample_corpus.txt`. This file contains standard sentences describing Word2Vec, CBOW, and PyTorch, which will serve as our training data.


In [2]:
# Check paths for both standalone running and repository-based running
corpus_path = "sample_corpus.txt"
if not os.path.exists(corpus_path):
    corpus_path = os.path.join("CBOW_PyTorch", "sample_corpus.txt")

if not os.path.exists(corpus_path):
    raise FileNotFoundError(f"Corpus file not found at: {corpus_path}")

print(f"Reading corpus from: {corpus_path}")
with open(corpus_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Raw Corpus Content:")
print("-" * 40)
print(raw_text)
print("-" * 40)


Reading corpus from: sample_corpus.txt
Raw Corpus Content:
----------------------------------------
Continuous Bag-of-Words (CBOW) is a popular model architecture for training word embeddings.
Word embeddings are dense vector representations of words in a continuous vector space where semantically similar words are mapped to nearby points.
In the CBOW architecture, the model predicts a target word from a set of surrounding context words.
This is in contrast to the Skip-Gram architecture, which predicts the surrounding context words given a single target word.
PyTorch is a powerful deep learning library that provides tools like embedding layers to implement these architectures easily.
We can train a CBOW model using a small corpus of text to learn word vectors.
Once trained, the model can predict the most likely target word given some context words.
Word embeddings are foundational for many natural language processing tasks such as sentiment analysis, machine translation, and language m

## Text Preprocessing

Before building a vocabulary, we need to clean and tokenize the raw text. Preprocessing converts all text to lowercase, replaces carriage returns/newlines/tabs/hyphens with spaces, removes other punctuation characters, and splits the clean string into a list of individual word tokens.


In [3]:
def preprocess_text(text: str) -> List[str]:
    """
    Cleans and tokenizes raw input text.
    """
    # Convert to lowercase
    text = text.lower()
    
    # Replace hyphens, carriage returns, newlines, and tabs with a space
    text = re.sub(r'[\r\n\t\-]', ' ', text)
    
    # Remove punctuation, keeping alphanumeric characters and spaces
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # Split by any whitespace sequence to tokenize
    tokens = text.split()
    return tokens

tokens = preprocess_text(raw_text)
print(f"Total tokens in preprocessed corpus: {len(tokens)}")
print("First 15 tokens:", tokens[:15])


Total tokens in preprocessed corpus: 141
First 15 tokens: ['continuous', 'bag', 'of', 'words', 'cbow', 'is', 'a', 'popular', 'model', 'architecture', 'for', 'training', 'word', 'embeddings', 'word']


## Vocabulary Creation

To train a neural network on text, we must represent words as numerical indices. We build two dictionary mappings:
1. `word_to_ix`: Maps each unique word string to a unique integer index.
2. `ix_to_word`: Maps each unique integer index back to its word string.

The vocabulary size determines the number of unique words the model can understand.


In [4]:
def build_vocab(tokens: List[str]) -> Tuple[Dict[str, int], Dict[int, str]]:
    """
    Builds word-to-index and index-to-word mappings from a list of tokens.
    """
    unique_tokens = sorted(list(set(tokens)))
    word_to_ix = {word: i for i, word in enumerate(unique_tokens)}
    ix_to_word = {i: word for i, word in enumerate(unique_tokens)}
    return word_to_ix, ix_to_word

word_to_ix, ix_to_word = build_vocab(tokens)
vocab_size = len(word_to_ix)
print(f"Vocabulary Size: {vocab_size} unique words.")
print("Sample word-to-index mappings:", list(word_to_ix.items())[:10])


Vocabulary Size: 85 unique words.
Sample word-to-index mappings: [('a', 0), ('analysis', 1), ('and', 2), ('architecture', 3), ('architectures', 4), ('are', 5), ('as', 6), ('bag', 7), ('can', 8), ('cbow', 9)]


## Context-Target Pair Generation

We generate training samples using a sliding window. For each word (the target), we extract $C$ words to its left and $C$ words to its right (the context). Boundary words that do not have enough left or right context are skipped to maintain a constant context size of $2 \times C$.


In [5]:
def generate_context_target_pairs(tokens: List[str], window_size: int = 2) -> List[Tuple[List[str], str]]:
    """
    Generates training pairs for the CBOW model using a sliding window.
    """
    pairs = []
    if len(tokens) <= 2 * window_size:
        return pairs
        
    for i in range(window_size, len(tokens) - window_size):
        context = tokens[i - window_size : i] + tokens[i + 1 : i + 1 + window_size]
        target = tokens[i]
        pairs.append((context, target))
    return pairs

window_size = 2
pairs = generate_context_target_pairs(tokens, window_size=window_size)
print(f"Generated {len(pairs)} context-target pairs.")
print("Sample Pair (Context -> Target):")
if pairs:
    print(f"{pairs[0][0]} -> {pairs[0][1]}")


Generated 137 context-target pairs.
Sample Pair (Context -> Target):
['continuous', 'bag', 'words', 'cbow'] -> of


## Dataset

We wrap our context-target pairs in a custom PyTorch `Dataset` class (`CBOWDataset`). For a given sample index, it retrieves the context words and the target word, maps them to their respective vocabulary indices, and returns them as LongTensors.
We then load this dataset into a PyTorch `DataLoader` to handle batching and shuffling.


In [6]:
class CBOWDataset(Dataset):
    """
    Custom PyTorch Dataset for the Continuous Bag-of-Words (CBOW) model.
    """
    def __init__(self, pairs: List[Tuple[List[str], str]], word_to_ix: Dict[str, int]):
        self.pairs = pairs
        self.word_to_ix = word_to_ix

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        context_words, target_word = self.pairs[idx]
        context_indices = [self.word_to_ix[w] for w in context_words]
        target_index = self.word_to_ix[target_word]
        
        context_tensor = torch.tensor(context_indices, dtype=torch.long)
        target_tensor = torch.tensor(target_index, dtype=torch.long)
        return context_tensor, target_tensor

# Instantiate dataset and dataloader
batch_size = 32
dataset = CBOWDataset(pairs, word_to_ix)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"Dataset successfully created. Total batches: {len(dataloader)}")


Dataset successfully created. Total batches: 5


## Model Definition

We define the `CBOWModel` using PyTorch's `nn.Module`:
1. **Embedding Layer**: Maps word indices to dense vectors of shape `[batch_size, context_size, embedding_dim]`.
2. **Mean Pooling**: Averages the embeddings of the context words along the context dimension to produce a single aggregated vector of shape `[batch_size, embedding_dim]`.
3. **Linear Layer**: Projects this averaged vector onto the vocabulary space to produce logits of shape `[batch_size, vocab_size]`.


In [7]:
class CBOWModel(nn.Module):
    """
    Continuous Bag-of-Words (CBOW) Model implemented in PyTorch.
    """
    def __init__(self, vocab_size: int, embedding_dim: int):
        super(CBOWModel, self).__init__()
        # Embedding lookup layer
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Output projection layer
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # Retrieve word embeddings: [batch_size, context_size, embedding_dim]
        embeds = self.embeddings(inputs)
        # Average embeddings along context dimension: [batch_size, embedding_dim]
        mean_embeds = torch.mean(embeds, dim=1)
        # Project to vocabulary space logits: [batch_size, vocab_size]
        logits = self.linear(mean_embeds)
        return logits

# Instantiate the model
embed_dim = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CBOWModel(vocab_size=vocab_size, embedding_dim=embed_dim).to(device)
print(model)


CBOWModel(
  (embeddings): Embedding(85, 50)
  (linear): Linear(in_features=50, out_features=85, bias=True)
)


## Training

We set up our training hyperparameters, the Cross-Entropy loss function, and the Adam optimizer.
We then run the training loop for a specified number of epochs, performing forward passes, computing the loss, performing backward passes to compute gradients, and updating model weights.


In [8]:
# Hyperparameters
epochs = 100
lr = 0.005

loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

print("Starting training loop...")
model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for context_batch, target_batch in dataloader:
        # Move batches to device
        context_batch = context_batch.to(device)
        target_batch = target_batch.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(context_batch)
        
        # Loss computation
        loss = loss_function(logits, target_batch)
        
        # Backward pass & weight optimization
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(target_batch)
        
    epoch_loss = total_loss / len(dataset)
    if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
        print(f"Epoch {epoch:03d}/{epochs:03d} - Loss: {epoch_loss:.4f}")
        
print("Training completed successfully!")


Starting training loop...
Epoch 001/100 - Loss: 4.4690
Epoch 010/100 - Loss: 2.7488
Epoch 020/100 - Loss: 1.4030
Epoch 030/100 - Loss: 0.6857
Epoch 040/100 - Loss: 0.3597
Epoch 050/100 - Loss: 0.2113
Epoch 060/100 - Loss: 0.1368
Epoch 070/100 - Loss: 0.0959
Epoch 080/100 - Loss: 0.0713
Epoch 090/100 - Loss: 0.0552
Epoch 100/100 - Loss: 0.0441
Training completed successfully!


## Save Model

After training, we save the model state dictionary containing the trained weights, alongside the metadata containing vocabulary mappings, configuration details, and hyperparameters. This allows us to load the model later for inference without retraining.


In [9]:
model_dir = "saved_model"
os.makedirs(model_dir, exist_ok=True)

# Save model weights
model_path = os.path.join(model_dir, "cbow_model.pt")
torch.save(model.state_dict(), model_path)
print(f"Trained model weights saved to: {model_path}")

# Save vocabulary metadata
metadata = {
    "word_to_ix": word_to_ix,
    "ix_to_word": {int(k): v for k, v in ix_to_word.items()},
    "window_size": window_size,
    "embed_dim": embed_dim,
    "vocab_size": vocab_size
}
metadata_path = os.path.join(model_dir, "metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)
print(f"Model metadata saved to: {metadata_path}")


Trained model weights saved to: saved_model/cbow_model.pt
Model metadata saved to: saved_model/metadata.json


## Inference

We write an inference helper function `predict_missing_word` to predict the target word from context words.
The function:
1. Tokenizes and cleans the context string.
2. Filters out any out-of-vocabulary (OOV) words.
3. Validates and adjusts the context length to match the expected $2 \times C$ inputs.
4. Runs the forward pass on the trained model.
5. Computes probabilities using Softmax and prints the top-k predictions.


In [10]:
def predict_missing_word(context_words: str, model: CBOWModel, metadata: dict, device: torch.device, top_k: int = 3):
    """
    Predicts the target word based on context words.
    """
    word_to_ix = metadata["word_to_ix"]
    ix_to_word = metadata["ix_to_word"]
    window_size = metadata["window_size"]
    expected_len = 2 * window_size
    
    # Clean and preprocess inputs
    tokens = preprocess_text(context_words)
    
    if not tokens:
        print("Error: No words entered or all words were removed by preprocessing.")
        return
        
    # Convert words to vocabulary indices
    indices = []
    oov_words = []
    for word in tokens:
        if word in word_to_ix:
            indices.append(word_to_ix[word])
        else:
            oov_words.append(word)
            
    if oov_words:
        print(f"Warning: Words {oov_words} are out of vocabulary (OOV) and were ignored.")
        
    if len(indices) == 0:
        print("Error: None of the entered words exist in the model's vocabulary.")
        return
        
    # Adjust context size to match window size
    if len(indices) != expected_len:
        print(f"Warning: Model expects exactly {expected_len} context words (window_size={window_size}). "
              f"You provided {len(indices)} valid words. Adjusting input size...")
        if len(indices) > expected_len:
            indices = indices[:expected_len]
        else:
            pad_ix = indices[0]
            indices += [pad_ix] * (expected_len - len(indices))
            
    print(f"Active context tokens: {[ix_to_word[idx] for idx in indices]}")
    
    # Run inference forward pass
    context_tensor = torch.tensor([indices], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(context_tensor)
        probabilities = F.softmax(logits, dim=1).squeeze(0)
        
    # Retrieve top predictions
    top_probs, top_indices = torch.topk(probabilities, k=min(top_k, len(probabilities)))
    
    print("\n--- Predictions ---")
    for i, (prob, idx_tensor) in enumerate(zip(top_probs, top_indices)):
        idx = idx_tensor.item()
        word = ix_to_word[idx]
        is_top = "(Best Predict)" if i == 0 else ""
        print(f"Rank {i+1}: {word:<15} Prob: {prob.item():.4f} {is_top}")
    print("-------------------\n")


## Sample Predictions

To verify our model works end-to-end, we load the saved model state dict and vocabulary metadata from disk. We then query the model to predict the word in the middle of context phrases.


In [11]:
# Load model and metadata from disk
with open(metadata_path, "r", encoding="utf-8") as f:
    loaded_metadata = json.load(f)
    
# Standardize keys (ensure integer keys for indexing)
loaded_metadata["ix_to_word"] = {int(k): v for k, v in loaded_metadata["ix_to_word"].items()}

loaded_model = CBOWModel(vocab_size=loaded_metadata["vocab_size"], embedding_dim=loaded_metadata["embed_dim"])
loaded_model.load_state_dict(torch.load(model_path, map_location=device))
loaded_model.to(device)
loaded_model.eval()
print("Model loaded successfully for prediction verification.\n")

# Define and run sample queries
sample_context = "the cbow the model"
print(f"Query Context: '{sample_context}'")
predict_missing_word(sample_context, loaded_model, loaded_metadata, device, top_k=3)


Model loaded successfully for prediction verification.

Query Context: 'the cbow the model'
Active context tokens: ['the', 'cbow', 'the', 'model']

--- Predictions ---
Rank 1: architecture    Prob: 0.9154 (Best Predict)
Rank 2: can             Prob: 0.0256 
Rank 3: predicts        Prob: 0.0106 
-------------------



## Conclusion

In this notebook, we successfully built, trained, and evaluated a Continuous Bag-of-Words (CBOW) model in PyTorch:
- We loaded and preprocessed a simple corpus, converting raw text to cleaned word tokens.
- We constructed the vocabulary word mappings (`word_to_ix` and `ix_to_word`).
- We implemented sliding window dataset formulation and loaded it via PyTorch's custom `Dataset` and `DataLoader`.
- We created the `CBOWModel` and trained it using Cross-Entropy loss.
- Finally, we saved the trained model and verified its prediction performance through custom inference tests.

This establishes a robust foundation for learning word embeddings and understanding representation learning in natural language processing!
